In [ ]:
import os
# import json
import itertools

import dask.dataframe as dd
import geopandas as gpd
import numpy as np
import pandas as pd

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns

# from sklearn.metrics import root_mean_squared_error

from scipy.stats import ks_2samp
from statsmodels.stats.multitest import multipletests

### Functions

In [ ]:
def getFiles(filepath):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
            # if not 'C2' in file:
            files.append(os.path.join(filepath, file))
    return files

In [ ]:
def plot_boxes(df, data_var, metric, palette, data_title, data_xlabel, data_ylabel, save_dir):
    # 1. Count observations for each unique combination of 'Bin' and 'xtrk_bin'
    group_counts = df.groupby(['Bin', data_var], observed=False)[metric].transform('count')
    
    # 2. Filter the DataFrame to keep only groups with more than 20 observations
    df_filtered = df[group_counts > 20]
    
    fig, ax = plt.subplots(figsize=(16, 10))
    sns.boxplot(data=df_filtered, x='Bin', y=metric,
                hue=data_var, palette=palette, ax=ax,
                linecolor='k', linewidth=2, showcaps=False)
    
    # --- Add Sample Sizes For Each Bar ---
    # Calculate precise counts for every unique combination
    counts = df_filtered.groupby(['Bin', data_var], observed=False).size().to_dict()
    
    # Get ordering used by Seaborn
    x_labels = [label.get_text() for label in ax.get_xticklabels()]
    hue_labels = [label.get_text() for label in ax.get_legend().get_texts()]
    
    # Calculate width offsets for the individual bars
    n_hues = len(hue_labels)
    box_width = 0.8  # Default Seaborn boxplot width
    offset_step = box_width / n_hues
    start_offset = -box_width / 2 + offset_step / 2
    
    # Loop through bins and hues to place text over each bar
    for x_idx, x_val in enumerate(x_labels):
        for h_idx, h_val in enumerate(hue_labels):
            # Handle numeric/string key conversions dynamically
            try:
                dict_key = (type(df_filtered['Bin'].iloc[0])(x_val), type(df_filtered[data_var].iloc[0])(h_val))
            except ValueError:
                dict_key = (x_val, h_val)
                
            count = counts.get(dict_key, 0)
            
            # Calculate individual X position for this specific bar
            bar_x = x_idx + start_offset + (h_idx * offset_step)
            
            # Only print if data exists for this specific sub-bar
            if count > 0 and metric == 'correlation':
                ax.text(bar_x, 1.03, f'{count}', ha='center', va='center', 
                        fontsize=11, fontweight='bold',
                        # rotation=90,
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

            elif count > 0 and metric == 'bias_abs':
                ax.text(bar_x, 0.765, f'{count}', ha='center', va='center', 
                        fontsize=11, fontweight='bold',
                        # rotation=90,
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
    # -------------------------------------
    if metric == 'correlation':
        plt.axhline(y=0.6, linestyle='--', color='k')
        plt.ylim(-0.2, 1)
    elif metric == 'bias_abs':
        plt.axhline(y=0.3, linestyle='--', color='k')
        plt.ylim(0, 0.75)
        
    plt.legend(title=data_title, fontsize=12, title_fontsize=14)
    plt.xlabel(data_xlabel, fontsize=16)
    plt.ylabel(data_ylabel, fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.savefig(os.path.join(save_dir, f'{data_var}_{metric}_box.png'), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def calc_KS(df, data_var, metric):

    # 1. Clean the data
    df_clean = df.dropna(subset=['Bin', data_var, metric])
    
    # 2. Container for results
    results = []
    
    # 3. Group by 'Bin'
    for bin_name, group in df_clean.groupby('Bin', observed=False):
    
        # Filter out xtrk_bins within this Bin that have fewer than 20 observations
        valid_data_bins = []
        for bin_, sub_group in group.groupby(data_var, observed=False):
            if len(sub_group) >= 20:
                valid_data_bins.append(bin_)
    
        # Need at least two valid xtrk_bins to perform a pairwise test
        if len(valid_data_bins) < 2:
            continue
    
        # 4. Generate pairs of valid xtrk_bins
        for bin_a, bin_b in itertools.combinations(valid_data_bins, 2):
            data_a = group[group[data_var] == bin_a][metric]
            data_b = group[group[data_var] == bin_b][metric]
    
            # 5. Run the KS test
            stat, p_val = ks_2samp(data_a, data_b)
    
            results.append(
                {
                    'Bin': bin_name,
                    'bin_A': bin_a,
                    'bin_B': bin_b,
                    'N_A': len(data_a),
                    'N_B': len(data_b),
                    'KS_Statistic': stat,
                    'p_value': p_val,
                }
            )
    
    # 6. Convert to DataFrame
    ks_metric_df = pd.DataFrame(results)
    
    # 7. Apply Benjamini-Hochberg (FDR) p-value correction if tests were performed
    if not ks_metric_df.empty:
        # returns: (rejected_null_boolean, corrected_p_values, _, _)
        _, corrected_pvals, _, _ = multipletests(
            ks_metric_df['p_value'], alpha=0.05, method='fdr_bh'
        )
        ks_metric_df['corrected_p_value'] = corrected_pvals
        ks_metric_df['significant'] = ks_metric_df['corrected_p_value'] < 0.05
        return ks_metric_df

In [ ]:
def plot_KS(df, data_var, metric, data_title, save_dir):
    # All unique labels for establishing matrix structure
    all_data_bins = sorted(
        list(
            set(df["bin_A"]).union(set(df["bin_B"]))
        )
    )
    unique_bins = ["(10, 30]", "(30, 50]", "(50, 380]"]
    
    # 2. Setup colors based on your style guidelines
    colors = ["lemonchiffon", "royalblue"]  # [Not Sig, Significant]
    cmap = sns.color_palette(colors)
    
    # Plot 3 horizontal subplots side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True, sharey=True)
    
    for i, bin_name in enumerate(unique_bins):
        ax = axes[i]
        sub_df = df[df["Bin"] == bin_name]
    
        # Initialize full, square p-value matrix template
        p_matrix = pd.DataFrame(
            np.nan, index=all_data_bins, columns=all_data_bins
        )
    
        # Populate bidirectional matrix entries to handle indexing seamlessly
        for _, row in sub_df.iterrows():
            p_matrix.loc[row["bin_A"], row["bin_B"]] = row[
                "corrected_p_value"
            ]
            p_matrix.loc[row["bin_B"], row["bin_A"]] = row[
                "corrected_p_value"
            ]
    
        # Categorical classification structure: 1 for Sig, 0 for Not Sig
        sig_matrix = (p_matrix < 0.05).astype(float)
    
        # Mask out upper triangle AND completely empty rows/columns (NaNs) to isolate the corner
        # np.triu with diagonal=0 hides upper half and the center line completely
        mask = np.triu(np.ones_like(p_matrix, dtype=bool), k=0) | p_matrix.isna()
    
        # Pass classified binary data to style background, but original values to 'annot'
        sns.heatmap(
            sig_matrix,
            mask=mask,
            annot=p_matrix,
            fmt=".3f",
            cmap=cmap,
            cbar=False,
            # linewidths=1,
            linecolor="#1c2833",  # Clean borders matching dark theme
            square=True,
            ax=ax,
            annot_kws={
                "size": 10,
                "weight": "bold",
            },  # Makes text readable over background colors
        )
    
        ax.set_title(f"Bin: {bin_name}", fontsize=13, fontweight="bold")
    
        # Updated dynamically formatted subplot headers
        ax.set_title(
            f"River width: {bin_name} m", fontsize=13, fontweight="bold"
        )
        ax.set_xlabel(data_title, fontsize=14)
        ax.tick_params(axis='x', labelsize=12)
        if i == 0:
            ax.set_ylabel(data_title, fontsize=14)
            ax.tick_params(axis='y', labelsize=12)
    
    # 3. Clean manual legend layout placement above subplots
    not_sig_patch = mpatches.Patch(
        color=colors[0], label="Not Significant (p ≥ 0.05)"
    )
    sig_patch = mpatches.Patch(color=colors[1], label="Significant (p < 0.05)")
    
    fig.legend(
        handles=[not_sig_patch, sig_patch],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.1),
        ncol=2,
        fontsize=12,
        frameon=False,
        prop={'size': 12, 'weight': 'semibold'}
    );
    
    fig.subplots_adjust(wspace=0.05);
    plt.savefig(os.path.join(save_dir, f'{data_var}_{metric}_ks.png'), dpi=300, bbox_inches='tight')
    plt.show()

### Setup

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/data/'
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results_figures/'

In [ ]:
# Order of all possible bins for plotting
order = ['(10, 30]', '(30, 50]', '(50, 380]']
# Cutpoints
bins = [10, 30, 50, 380]

# Order of cross-track bins
xtrk = [10, 20, 30, 40, 50, 60]
xtrk_order = ['(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]', '(50, 60]']

# Order of azimuth bins
azimuth_bins = [0, 10, 80, 100, 170, 190, 260, 280, 350, 360]
labels = ['parallel', 'other', 'perpendicular',
          'other', 'parallel', 'other',
          'perpendicular', 'other', 'parallel']

## Prep data

In [ ]:
output_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_output/PIXC_vD_wse_2026_06_01_rel_abs/'
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/data/'

# Get list of files
files = getFiles(output_dir)
print(len(files))

# Read in all files
ddf = dd.read_parquet(files)

# Bin reaches by width for pre-filtering analysis
ddf['Bin'] = ddf['WidthM'].map_partitions(pd.cut, bins=bins).astype(str)
ddf = ddf.reset_index(drop=True)

df = ddf.compute()
df = df[df['WidthM'] >= 10]

df['Bin'] = pd.Categorical(df['Bin'], categories=order, ordered=True)

print(len(df['site_id'].unique()))
print(len(df[~df.wse_gauge.isna()]['site_id'].unique()))

### Deal with cross-track

In [ ]:
df['xtrk_med'] = df['xtrk_med'] / 1000

df['xtrk_bin'] = pd.cut(df['xtrk_med'], bins=xtrk).astype(str)
df = df.reset_index(drop=True)
df['xtrk_bin'] = pd.Categorical(df['xtrk_bin'], categories=xtrk_order, ordered=True)

df_check = df[['site_id', 'xtrk_med', 'xtrk_bin']]
df_check = df_check[~df_check['xtrk_bin'].isna()]

df_grouped = df_check.groupby('site_id')['xtrk_med'].unique().reset_index()
df_grouped['min_val'] = df_grouped['xtrk_med'].apply(np.min)
df_grouped['max_val'] = df_grouped['xtrk_med'].apply(np.max)

df_grouped['range'] = df_grouped['max_val'] - df_grouped['min_val']
multi_xtrk_ids = df_grouped[df_grouped['range'] > 20]['site_id']

### Make main df to track stats

In [ ]:
df['xtrk_med'] = np.abs(df['xtrk_med'])
xtrk_med = df.groupby('site_id')['xtrk_med'].median()

df_main = df.drop_duplicates('site_id')[['site_id', 'dec_lat_va', 'dec_long_va',
                                         'nadir_az',
                                         'alt_va', 'datum_offset_m', 'lon_wgs84',
                                         'lat_wgs84', 'NHDPlusID', 'WidthM', 'new_width',
                                         'huc2', 'huc4', 'Bin']]

df_main = df_main.merge(xtrk_med, how='inner', on='site_id')

### Find number of obs per site with sufficient, useable data

In [ ]:
df_clean = df[df['n_pixels'] >= 5].copy()
len(df_clean['site_id'].unique())

num_obs = df_clean.groupby('site_id').size().reset_index().rename(columns={0: 'num_use_obs'})

# Merge on number of obs
df_main = df_main.merge(num_obs, how='left', on='site_id')

### Find bias for gauges with sufficient data

In [ ]:
# Get ids of gauges with >= 20 obs
use_ids = num_obs[num_obs['num_use_obs'] >= 20]['site_id']

df_agree = df_clean[df_clean['site_id'].isin(use_ids)]
len(df_agree['site_id'].unique())

# Calculate various error terms
groups = df_agree.groupby('site_id')

df_agree = df_agree.assign(
    wse_gauge_norm = df_agree['wse_gauge'] - groups['wse_gauge'].transform('median'),
    wse_swot_norm  = df_agree['wse_swot'] - groups['wse_swot'].transform('median')
)

df_agree['wse_bias'] = df_agree['wse_swot'] - df_agree['wse_gauge']
df_agree['wse_diff'] = df_agree['wse_swot_norm'] - df_agree['wse_gauge_norm']
df_agree['wse_diff_abs'] = df_agree['wse_diff'].abs()

# Final filtering
df_agree = df_agree[df_agree['wse_diff_abs'] < 5].copy()

df_bias = df_agree.groupby('site_id')['wse_diff_abs'].median().reset_index().rename(columns={'wse_diff_abs': 'bias_abs'})
agree_ids = df_bias[df_bias['bias_abs'] < 0.3]['site_id']

# Merge on bias
df_main = df_main.merge(df_bias, how='left', on='site_id')

### Find correlation for gauges with agreeable SWOT data

In [ ]:
df_agree_1 = df_agree[df_agree['site_id'].isin(agree_ids)].reset_index()

df_sense = df_agree_1.groupby('site_id', observed=False)[['wse_gauge_norm', 'wse_swot_norm']].corr()

df_sense = df_sense.unstack().iloc[:, 1].reset_index()
df_sense.columns = ['site_id', 'correlation']

# Merge on correlation
df_main = df_main.merge(df_sense, how='left', on='site_id')

## Cross-track distance

In [ ]:
df_main['xtrk_bin'] = pd.cut(df_main['xtrk_med'], bins=xtrk).astype(str)
df_main = df_main.reset_index(drop=True)
df_main['xtrk_bin'] = pd.Categorical(df_main['xtrk_bin'], categories=xtrk_order, ordered=True)

df_xtrk_plot = df_main[['site_id', 'NHDPlusID', 'WidthM', 'Bin', 'xtrk_med', 'xtrk_bin',
                   'num_use_obs', 'bias_abs', 'correlation']].copy()

df_xtrk_plot = df_xtrk_plot[~df_xtrk_plot['xtrk_bin'].isna()]

df_xtrk_plot = df_xtrk_plot[~df_xtrk_plot['site_id'].isin(multi_xtrk_ids)]

df_xtrk_plot = df_xtrk_plot[(df_xtrk_plot['num_use_obs'] >= 20) & ~(df_xtrk_plot['xtrk_bin'].isna())]

In [ ]:
plot_boxes(df=df_xtrk_plot,
            data_var='xtrk_bin',
            metric='bias_abs',
            palette='viridis',
            data_title='Cross-track distance (m)',
            data_xlabel='River width (m)',
            data_ylabel='Unsigned relative bias (m)',
          save_dir=save_dir)

In [ ]:
ks_bias_df = calc_KS(df=df_xtrk_plot,
                     data_var='xtrk_bin',
                     metric='bias_abs')

plot_KS(df=ks_bias_df,
        data_var='xtrk_bin',
        metric='bias_abs',
        data_title='Cross-track distance (m)',
        save_dir=save_dir)

In [ ]:
plot_boxes(df=df_xtrk_plot,
           data_var='xtrk_bin',
           metric='correlation',
           palette='viridis',
           data_title='Cross-track distance (m)',
           data_xlabel='River width (m)',
           data_ylabel='Correlation',
          save_dir=save_dir)

In [ ]:
ks_corr_df = calc_KS(df=df_xtrk_plot,
                     data_var='xtrk_bin',
                     metric='correlation')

plot_KS(df=ks_corr_df,
        data_var='xtrk_bin',
        metric='correlation',
        data_title='Cross-track distance (m)',
        save_dir=save_dir)

## Landcover

In [ ]:
gauges = pd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations_with_stage_and_nhd_metadata_az_div_nlcd.parquet')

df_land = df_main[['site_id', 'NHDPlusID', 'WidthM', 'Bin',
                   'num_use_obs', 'bias_abs', 'correlation']].copy()
df_land = df_land.merge(gauges[['NHDPlusID', 'mode_land']], how='left', on='NHDPlusID')

df_land_plot = df_land[~df_land['mode_land'].isna()].copy()
df_land_plot['mode_land_parent'] = df_land_plot['mode_land'] // 10

In [ ]:
nlcd_level1_palette = {
    # 1: "#466b9f",  # Water & Ice
    'Developed': "#ff0000",  # Developed
    'Barren': "#d2cdc0",  # Barren
    'Forest': "#38814e",  # Forest
    'Shrubland': "#DCCA8F",  # Shrubland
    'Herbaceous': "#A3CC51",  # Herbaceous
    'Planted/Cultivated': "#CA9146",  # Agricultural
    'Wetlands': "#64B3D5"   # Wetlands
}

labels_nlcd = [
    # "Water",
    "Developed",
    "Barren",
    "Forest",
    "Shrubland",
    "Herbaceous",
    "Planted/Cultivated",
    "Wetlands"
]

sorted_categories = sorted(df_land_plot['mode_land_parent'].unique())

In [ ]:
category_map = dict(zip(sorted_categories, labels_nlcd))
df_land_plot['mode_land_name'] = df_land_plot['mode_land_parent'].map(category_map)

parent_order = sorted(df_land_plot['mode_land_parent'].unique())
name_order = labels_nlcd

parent_dtype = pd.CategoricalDtype(categories=parent_order, ordered=True)
name_dtype = pd.CategoricalDtype(categories=name_order, ordered=True)

df_land_plot['mode_land_parent'] = df_land_plot['mode_land_parent'].astype(parent_dtype)
df_land_plot['mode_land_name'] = df_land_plot['mode_land_name'].astype(name_dtype)

In [ ]:
plot_boxes(df=df_land_plot,
           data_var='mode_land_name',
           metric='bias_abs',
           palette=nlcd_level1_palette,
           data_title='Landcover',
           data_xlabel='River width (m)',
           data_ylabel='Unsigned relative bias (m)',
           save_dir=save_dir)

In [ ]:
ks_bias_df = calc_KS(df=df_land_plot,
                     data_var='mode_land_name',
                     metric='bias_abs')

plot_KS(df=ks_bias_df,
        data_var='mode_land_name',
        metric='bias_abs',
        data_title='Landcover',
        save_dir=save_dir)

In [ ]:
plot_boxes(df=df_land_plot,
           data_var='mode_land_name',
           metric='correlation',
           palette=nlcd_level1_palette,
           data_title='Landcover',
           data_xlabel='River width (m)',
           data_ylabel='Correlation',
           save_dir=save_dir)

In [ ]:
ks_corr_df = calc_KS(df=df_land_plot,
                     data_var='mode_land_name',
                     metric='correlation')

plot_KS(df=ks_corr_df,
        data_var='mode_land_name',
        metric='correlation',
        data_title='Landcover',
        save_dir=save_dir)

## Alignment

In [ ]:
gauges = pd.read_parquet('/nas/cee-water/cjgleason/fiona/USGS_gauges/usgs_stations_with_stage_and_nhd_metadata_azimuth.parquet')

df_align = df_main[['site_id', 'NHDPlusID', 'WidthM', 'Bin', 'nadir_az',
                   'num_use_obs', 'bias_abs', 'correlation']].copy()
df_align = df_align.merge(gauges[['NHDPlusID', 'Azimuth']], how='left', on='NHDPlusID')

## Calculate aligment of nadir and reach
df_align['alignment'] = df_align['Azimuth'] - df_align['nadir_az']

# Handle when angle < 0
df_align['alignment'] = np.where(df_align['alignment'] < 0,
                                 df_align['alignment'] + 360,
                                 df_align['alignment'])

df_align['align_bin'] = pd.cut(df_align['alignment'], bins=azimuth_bins, labels=labels, ordered=False).astype(str)

df_align_plot = df_align[~df_align['alignment'].isna()].copy()

In [ ]:
plot_boxes(df=df_align_plot,
                data_var='align_bin',
                metric='bias_abs',
                palette='plasma',
                data_title='Reach-nadir alignment',
                data_xlabel='River width (m)',
                data_ylabel='Unsigned relative bias (m)',
           save_dir=save_dir)

In [ ]:
ks_bias_df = calc_KS(df=df_align_plot,
                     data_var='align_bin',
                     metric='bias_abs')

plot_KS(df=ks_bias_df,
        data_var='align_bin',
        metric='bias_abs',
        data_title='Reach-nadir alignment',
        save_dir=save_dir)

In [ ]:
plot_boxes(df=df_align_plot,
           data_var='align_bin',
           metric='correlation',
            palette='plasma',
            data_title='Reach-nadir alignment',
            data_xlabel='River width (m)',
            data_ylabel='Correlation',
           save_dir=save_dir)

In [ ]:
ks_corr_df = calc_KS(df=df_align_plot,
                     data_var='align_bin',
                     metric='correlation')

plot_KS(df=ks_corr_df,
        data_var='align_bin',
        metric='correlation',
        data_title='Reach-nadir alignment',
        save_dir=save_dir)